# YouTube Channel Video Query & Filter

Query a YouTube channel for videos and filter them by keywords.

## Setup & Configuration

In [1]:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build
import pandas as pd

# Change the YouTube channel ID here
CHANNEL_ID = 'UCupvZG-5ko_eiXAupbDfxWw'  # CNN channel ID
MAX_VIDEOS = 50  # Set to desired limit (100-500 recommended)

# Load API key from .env file
# To resolve this error, please create a file named `.env` in the same directory as this notebook
# and add your YouTube Data API v3 key to it in the format:

YOUTUBE_API_KEY_NAME = 'YOUTUBE_API_KEY'

load_dotenv()
YOUTUBE_API_KEY = os.getenv(YOUTUBE_API_KEY_NAME)

# Alternatively, if you prefer not to use a .env file, uncomment the line below
# and replace 'YOUR_API_KEY_HERE' with your actual YouTube Data API v3 key.
# YOUTUBE_API_KEY = 'YOUR_API_KEY_HERE'

# Otherwise, if the API key is stored in Colab secrets, uncomment the 2 line below
# from google.colab import userdata
# YOUTUBE_API_KEY = userdata.get(YOUTUBE_API_KEY_NAME)

if not YOUTUBE_API_KEY:
    raise ValueError('YOUTUBE_API_KEY not found. Please follow the instructions above to set your API key.')

# Initialize YouTube API
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
print('YouTube API initialized successfully')

YouTube API initialized successfully


## Define Keywords to Filter

In [2]:
# Flattened list of keywords for filtering
FILTER_KEYWORDS = [
    'AI', 'artificial intelligence', 'machine learning',
    'medicine', 'medical', 'health', 'vaccine', 'vaccination',
    'earth', 'climate', 'environment', 'global warming',
    'space', 'NASA', 'astronomy',
    'science', 'research', 'study', 'scientist',
    'biology', 'physics', 'chemistry',
    'disease', 'pandemic', 'virus',
    'climate change', 'renewable energy', 'technology', 'tech', 'innovation',
    'politics', 'government', 'policy', 'election', 'president', 'congress', 'senate', 'war', 'conflict', 'vote', 'voting'
]

# YouTube category ID mappings
CATEGORY_ID_MAP = {
    '1': 'Film & Animation',
    '2': 'Autos & Vehicles',
    '10': 'Music',
    '15': 'Pets & Animals',
    '17': 'Sports',
    '18': 'Short Movies',
    '19': 'Travel & Events',
    '20': 'Gaming',
    '21': 'Videoblogging',
    '22': 'People & Blogs',
    '23': 'Comedy',
    '24': 'Entertainment',
    '25': 'News & Politics',
    '26': 'Howto & Style',
    '27': 'Education',
    '28': 'Science & Tech',
    '29': 'Nonprofits & Activism',
    '30': 'Movies',
    '31': 'Anime/Animation',
    '32': 'Action/Adventure',
    '33': 'Classics',
    '34': 'Comedies',
    '35': 'Documentaries',
    '36': 'Dramas',
    '37': 'Family',
    '38': 'Foreign',
    '39': 'Horror',
    '40': 'Sci-Fi/Fantasy',
    '41': 'Thrillers',
    '42': 'Shorts',
    '43': 'Shows',
    '44': 'Trailers'
}

print(f'Filtering by {len(FILTER_KEYWORDS)} keywords')

Filtering by 41 keywords


## Step 1: Get Videos from Channel

In [3]:
def get_channel_videos(channel_id, max_videos=100):
    """
    Get videos from a channel.
    Args:
        channel_id: YouTube channel ID
        max_videos: Maximum number of videos to fetch (default 100)
    Returns:
        list of video IDs
    """
    video_ids = []
    next_page_token = None
    
    # Get uploads playlist ID for the channel
    channel_response = youtube.channels().list(
        id=channel_id,
        part='contentDetails'
    ).execute()
    
    if not channel_response['items']:
        print(f'Channel {channel_id} not found')
        return video_ids
    
    uploads_playlist_id = channel_response['items'][0]['contentDetails']['relatedPlaylists']['uploads']
    
    # Get videos from uploads playlist
    while len(video_ids) < max_videos:
        playlist_response = youtube.playlistItems().list(
            playlistId=uploads_playlist_id,
            part='contentDetails',
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        
        for item in playlist_response.get('items', []):
            if len(video_ids) >= max_videos:
                break
            video_ids.append(item['contentDetails']['videoId'])
        
        next_page_token = playlist_response.get('nextPageToken')
        if not next_page_token:
            break
    
    return video_ids

print(f'Fetching up to {MAX_VIDEOS} videos from channel {CHANNEL_ID}...')
video_ids = get_channel_videos(CHANNEL_ID, max_videos=MAX_VIDEOS)
print(f'✓ Found {len(video_ids)} videos')

Fetching up to 50 videos from channel UCupvZG-5ko_eiXAupbDfxWw...
✓ Found 50 videos


## Step 2: Extract Metadata from Videos

In [4]:
def get_video_metadata(video_ids):
    """
    Get metadata for videos.
    Returns list of dicts with video info.
    """
    videos = []
    
    # Process in batches (API limit is 50 per request)
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        
        response = youtube.videos().list(
            id=','.join(batch),
            part='snippet,statistics'
        ).execute()
        
        for item in response['items']:
            snippet = item['snippet']
            stats = item['statistics']
            
            video = {
                'video_id': item['id'],
                'title': snippet['title'],
                'description': snippet['description'],
                'tags': snippet.get('tags', []),
                'category_id': snippet.get('categoryId'),
                'view_count': int(stats.get('viewCount', 0)),
                'like_count': int(stats.get('likeCount', 0)),
                'comment_count': int(stats.get('commentCount', 0))
            }
            videos.append(video)
    
    return videos

# Get metadata (only if we have video IDs)
if video_ids:
    videos = get_video_metadata(video_ids)
    df = pd.DataFrame(videos)
    print(f'Extracted metadata for {len(videos)} videos')
    df.head()
else:
    print('No video IDs to process')

Extracted metadata for 50 videos


In [5]:
import re

# Keywords that are too ambiguous/generic for description matching
STRICT_KEYWORDS = {
    'AI', 'tech', 'virus', 'disease', 'pandemic', 'science', 'research', 
    'study', 'health', 'medicine', 'earth', 'space', 'environment',
    'policy', 'conflict', 'politics', 'election'
}

def matches_keywords(video, keywords):
    """
    Check if video matches any of the keywords.
    Returns (matched_keyword, match_source) or (None, None).
    
    Matching rules:
    - Tags: Always checked (most reliable)
    - Title: Always checked (usually accurate)
    - Description: ONLY for keywords > 10 chars AND not in STRICT_KEYWORDS
    """
    tags_text = ' '.join(video.get('tags', [])).lower()
    title_text = video['title'].lower()
    description_text = video['description'].lower()
    
    for keyword in keywords:
        pattern = r'\b' + re.escape(keyword.lower()) + r'\b'
        
        # Priority 1: Check tags first (most reliable)
        if re.search(pattern, tags_text):
            return keyword, 'tags'
        
        # Priority 2: Check title (usually reliable)
        if re.search(pattern, title_text):
            return keyword, 'title'
        
        # Priority 3: Check description ONLY for specific keywords
        # Exclude STRICT keywords and short keywords
        if keyword not in STRICT_KEYWORDS and len(keyword) > 10:
            if re.search(pattern, description_text):
                return keyword, 'description'
    
    return None, None

# Filter videos with match reporting
filtered_videos = []
match_sources = []

for v in videos:
    matched_keyword, match_source = matches_keywords(v, FILTER_KEYWORDS)
    if matched_keyword:
        v['matched_keyword'] = matched_keyword
        v['category_name'] = CATEGORY_ID_MAP.get(v.get('category_id'), 'Unknown')
        filtered_videos.append(v)
        match_sources.append(match_source)

filtered_df = pd.DataFrame(filtered_videos)
filtered_df['match_source'] = match_sources

print(f'Matched {len(filtered_videos)} out of {len(videos)} videos\n')

# Show results with match source
display_columns = ['title', 'matched_keyword', 'match_source']
print('Filtered Results (with match source):')
print(filtered_df[display_columns].to_string(index=False))

Matched 21 out of 50 videos

Filtered Results (with match source):
                                                                   title matched_keyword match_source
                            Persian American Jews react to war with Iran             war        title
                         AI Images of Iran's new leader circulate online              AI        title
                 Seventh US service member killed in Iran war identified             war        title
               US responsible for Iran school strike, new video suggests             war         tags
              Senior Iranian official tells CNN: Iran ready for long war             war        title
                      Hours-long delays at airports due to TSA shortages      government         tags
                     Iran war: Now the biggest oil disruption in history       president         tags
                          Asia stocks plummet in second week of Iran war             war        title
               

## Optional: Save Results

In [6]:
# Save to CSV with timestamp
from datetime import datetime

# Create filename with timestamp
timestamp = datetime.now().strftime('%Y-%m-%d_%H%M%S')
filename = f'filtered_videos_{timestamp}.csv'

# Select columns to save
save_columns = ['title', 'matched_keyword', 'match_source', 'category_name', 'view_count', 'like_count', 'tags']

# Create a copy with Title Case column names
export_df = filtered_df[save_columns].copy()

# Convert tags list to comma-separated string for CSV readability
export_df['tags'] = export_df['tags'].apply(lambda x: ', '.join(x) if x else '')

column_mapping = {
    'title': 'Title',
    'matched_keyword': 'Matched Keyword',
    'match_source': 'Match Source',
    'category_name': 'Category Name',
    'view_count': 'View Count',
    'like_count': 'Like Count',
    'tags': 'Tags'
}

export_df.rename(columns=column_mapping, inplace=True)
export_df.to_csv(filename, index=False)

print(f'✓ Results saved to {filename}')
print(f'  Videos: {len(filtered_df)} matched')
print(f'  Columns: {", ".join(column_mapping.values())}')

✓ Results saved to filtered_videos_2026-03-09_223508.csv
  Videos: 21 matched
  Columns: Title, Matched Keyword, Match Source, Category Name, View Count, Like Count, Tags


In [7]:
# Analyze tag coverage across ALL videos
if 'df' in locals():
    videos_with_tags = df[df['tags'].apply(len) > 0].shape[0]
    videos_without_tags = df[df['tags'].apply(len) == 0].shape[0]
    
    print(f"Tag Coverage Analysis:")
    print(f"  Videos with tags: {videos_with_tags} ({videos_with_tags/len(df)*100:.1f}%)")
    print(f"  Videos without tags: {videos_without_tags} ({videos_without_tags/len(df)*100:.1f}%)")
    
    if videos_with_tags > 0:
        print(f"\nSample video with tags:")
        sample = df[df['tags'].apply(len) > 0].iloc[0]
        print(f"  Title: {sample['title']}")
        print(f"  Tags: {sample['tags']}")
    
    if videos_without_tags > 0:
        print(f"\nSample video without tags:")
        sample = df[df['tags'].apply(len) == 0].iloc[0]
        print(f"  Title: {sample['title']}")

Tag Coverage Analysis:
  Videos with tags: 30 (60.0%)
  Videos without tags: 20 (40.0%)

Sample video with tags:
  Title: Seventh US service member killed in Iran war identified
  Tags: ['#cnn', '#news']

Sample video without tags:
  Title: 'We are ready to sacrifice our blood': Iranians share message for Trump
